# 1. Setup and file discovery

This notebook audits the course `binSamples` and `fillSamples` data for the final Quantitative Trading and Price Impact project. It checks whether the data supports synthetic intraday alphas, strategy experiments, stress tests, and rolling-window extensions.

VS Code/Jupyter often runs notebooks with the working directory set to `price-impact-project/notebooks` rather than the repository root. The setup cell below resolves the project root by walking upward from `Path.cwd()`, adds it to `sys.path`, and then discovers CSV files under `data/` consistently.

Use `MAX_FILES`, `NROWS`, and `SELECTED_STOCKS` below to make quick test runs on a smaller subset.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


def find_project_root(start: Path) -> Path:
    """Find the repository root from either the repo root or notebooks directory."""

    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent

    for candidate in [start] + list(start.parents):
        has_project_markers = (
            (candidate / "data").exists()
            or (candidate / "src").exists()
            or (candidate / ".git").exists()
        )
        is_not_notebooks = candidate.name != "notebooks"
        if has_project_markers and is_not_notebooks:
            return candidate

    raise RuntimeError(
        "Could not find project root. Expected a parent directory containing data/, src/, or .git/."
    )


def discover_data_files(project_root: Path) -> tuple[list[Path], list[Path]]:
    """Discover bin and fill CSV files from expected folders, with recursive fallback."""

    data_dir = project_root / "data"
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    all_csvs = sorted(data_dir.rglob("*.csv"))
    bin_dir = data_dir / "binSamples"
    fill_dir = data_dir / "fillSamples"

    bin_files_found = sorted(bin_dir.glob("*.csv")) if bin_dir.exists() else []
    fill_files_found = sorted(fill_dir.glob("*.csv")) if fill_dir.exists() else []

    if not bin_files_found:
        bin_files_found = [
            path for path in all_csvs
            if "bin" in path.parent.name.lower() or "bin" in path.name.lower()
        ]
    if not fill_files_found:
        fill_files_found = [
            path for path in all_csvs
            if "fill" in path.parent.name.lower() or "fill" in path.name.lower()
        ]

    return bin_files_found, fill_files_found


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_audit_utils as dau

DATA_DIR = PROJECT_ROOT / "data"
BIN_DIR = DATA_DIR / "binSamples"
FILL_DIR = DATA_DIR / "fillSamples"
OUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MAX_FILES = 1
NROWS = 200_000
SELECTED_STOCKS = None
SAMPLE_N_STOCKS_FOR_PLOTS = 3
SAMPLE_N_DAYS_FOR_PLOTS = 2

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

all_data_csvs = sorted(DATA_DIR.rglob("*.csv")) if DATA_DIR.exists() else []
bin_files, fill_files = discover_data_files(PROJECT_ROOT)

print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())
print("BIN_DIR exists:", BIN_DIR.exists())
print("FILL_DIR exists:", FILL_DIR.exists())
print(f"Found {len(all_data_csvs)} total CSV files under data/")
print(f"Found {len(bin_files)} bin CSV files")
print(f"Found {len(fill_files)} fill CSV files")

print("Example bin files:")
for path in bin_files[:5]:
    print("  -", path.relative_to(PROJECT_ROOT))
print("Example fill files:")
for path in fill_files[:5]:
    print("  -", path.relative_to(PROJECT_ROOT))

display(dau.file_inventory(bin_files))
display(dau.file_inventory(fill_files))

In [ ]:
print("Current working directory:", Path.cwd().resolve())
print("Resolved project root:", PROJECT_ROOT)
print("\nProject root contents:")
for path in sorted(PROJECT_ROOT.iterdir()):
    print("  -", path.name + ("/" if path.is_dir() else ""))

print("\nData folder contents:")
if DATA_DIR.exists():
    for path in sorted(DATA_DIR.iterdir()):
        print("  -", path.relative_to(PROJECT_ROOT), "dir" if path.is_dir() else "file")
else:
    print("  data/ does not exist")

print("\nFirst 20 CSVs under data/:")
for path in all_data_csvs[:20]:
    print("  -", path.relative_to(PROJECT_ROOT))

# 2. Load data

Each CSV is loaded with a `source_file` column and a `sample_type` tag. If the raw data is too large for an exploratory run, set `MAX_FILES` or `NROWS` in the setup cell.

In [ ]:
bin_df = dau.load_csv_files(bin_files, "bin", max_files=MAX_FILES, nrows=NROWS)
fill_df = dau.load_csv_files(fill_files, "fill", max_files=MAX_FILES, nrows=NROWS)

for name, df in [("binSamples", bin_df), ("fillSamples", fill_df)]:
    print("" + "=" * 80)
    print(name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("dtypes:")
    print(df.dtypes)
    print("memory MB:", df.memory_usage(deep=True).sum() / 1024**2)
    if not df.empty:
        print("unique stocks:", df["stock"].nunique() if "stock" in df else "missing stock column")
        print("unique dates:", df["date"].nunique() if "date" in df else "missing date column")
        display(df.head())

# 3. Standardize date/time/timestamp

The project needs reliable within-stock and within-day ordering for future returns, signal delays, and forced liquidation. This section parses `date` and `time` into `date_parsed` and `timestamp`, then checks duplicate timestamps and monotonicity by stock-date.

In [ ]:
for name, df in [("binSamples", bin_df), ("fillSamples", fill_df)]:
    if not df.empty:
        print(f"{name} date examples:", df["date"].dropna().astype(str).head(10).tolist())
        print(f"{name} time examples:", df["time"].dropna().astype(str).head(10).tolist())

bin_df = dau.standardize_datetime(bin_df) if not bin_df.empty else bin_df
fill_df = dau.standardize_datetime(fill_df) if not fill_df.empty else fill_df

def timestamp_checks(df, name):
    if df.empty:
        return pd.DataFrame()
    key_cols = ["stock", "date_parsed", "timestamp"]
    duplicate_keys = int(df.duplicated(key_cols).sum())
    monotonic = df.groupby(["stock", "date_parsed"], sort=False)["timestamp"].apply(lambda s: s.is_monotonic_increasing)
    report = pd.DataFrame({
        "sample_type": [name],
        "duplicate_stock_date_timestamp_rows": [duplicate_keys],
        "n_stock_dates": [len(monotonic)],
        "n_non_monotonic_stock_dates": [int((~monotonic).sum())],
        "timestamp_missing_pct": [float(df["timestamp"].isna().mean())],
    })
    return report

timestamp_report = pd.concat([timestamp_checks(bin_df, "bin"), timestamp_checks(fill_df, "fill")], ignore_index=True)
display(timestamp_report)

bin_stock_date_counts = dau.stock_date_counts(bin_df)
fill_stock_date_counts = dau.stock_date_counts(fill_df)
bin_stock_date_counts.to_csv(OUT_DIR / "bin_stock_date_counts.csv", index=False)
fill_stock_date_counts.to_csv(OUT_DIR / "fill_stock_date_counts.csv", index=False)

display(bin_stock_date_counts.head())
display(fill_stock_date_counts.head())

if not bin_df.empty:
    display(bin_df.groupby("date_parsed").size().rename("rows").reset_index().head())
    display(bin_df.groupby("stock").size().rename("rows").sort_values(ascending=False).reset_index().head())

# 4. Schema validation

These checks verify whether the expected project columns exist, whether unexpected columns are present, and whether duplicates might interfere with groupby calculations or merges.

In [ ]:
bin_expected_columns = ["date", "time", "stock", "trade", "orderFlow", "hidden", "auction", "mid", "midEnd", "spread", "effSpread", "lobImb", "effLobImb", "trdLiq", "ofLiq", "depth", "nbEvents", "nbHidden", "nbTrades"]
fill_expected_columns = ["date", "stock", "time", "trade", "mid", "spread", "effSpread", "depth", "lobImb", "ask", "bid", "askVolume", "bidVolume"]

schema_bin = dau.schema_validation_report(bin_df, dau.BIN_REQUIRED_COLUMNS, bin_expected_columns + ["source_file", "sample_type", "date_parsed", "timestamp"])
schema_fill = dau.schema_validation_report(fill_df, dau.FILL_REQUIRED_COLUMNS, fill_expected_columns + ["source_file", "sample_type", "date_parsed", "timestamp"])

schema_bin.to_csv(OUT_DIR / "schema_validation_bin.csv", index=False)
schema_fill.to_csv(OUT_DIR / "schema_validation_fill.csv", index=False)

display(schema_bin)
display(schema_fill)

# 5. Basic data quality checks

These checks look for impossible or suspicious values in prices, spreads, depths, volume/count fields, imbalances, trades, and order flow. They are important because impact models and synthetic future returns are very sensitive to bad prices and malformed liquidity fields.

In [ ]:
bin_quality = dau.data_quality_checks(bin_df, "bin")
fill_quality = dau.data_quality_checks(fill_df, "fill")
display(bin_quality)
display(fill_quality)

bin_numeric_summary = dau.numeric_summary(bin_df)
fill_numeric_summary = dau.numeric_summary(fill_df)
bin_numeric_summary.to_csv(OUT_DIR / "bin_numeric_summary.csv", index=False)
fill_numeric_summary.to_csv(OUT_DIR / "fill_numeric_summary.csv", index=False)

display(bin_numeric_summary)
display(fill_numeric_summary)

for col in ["trade", "orderFlow"]:
    if col in bin_df.columns:
        print(f"{col} value summary")
        display(bin_df[col].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# 6. Missing values and duplicates

Missingness and duplicate key checks help decide whether the data can safely drive rolling calibration, future-return construction, and bin/fill merging.

In [ ]:
if not bin_df.empty:
    bin_missing = dau.save_missing_plot(bin_df, "binSamples missing values", FIG_DIR / "bin_missing_values.png")
    display(bin_missing)
    print("bin duplicate rows:", int(bin_df.duplicated().sum()))
    print("bin duplicate stock/date/time rows:", int(bin_df.duplicated(["stock", "date", "time"]).sum()))
if not fill_df.empty:
    fill_missing = dau.save_missing_plot(fill_df, "fillSamples missing values", FIG_DIR / "fill_missing_values.png")
    display(fill_missing)
    print("fill duplicate rows:", int(fill_df.duplicated().sum()))
    print("fill duplicate stock/date/time rows:", int(fill_df.duplicated(["stock", "date", "time"]).sum()))

# 7. Coverage and universe selection checks

The baseline project setup needs fitting on 20 stocks over one month and applying to the next month for the same 20 stocks. This section measures stock/month/date coverage and proposes a candidate 20-stock universe.

In [ ]:
monthly_availability = dau.monthly_availability(bin_df, fill_df)
monthly_availability.to_csv(OUT_DIR / "monthly_availability.csv", index=False)
display(monthly_availability)

candidate_universe_20 = dau.candidate_universe(bin_df, fill_df, n_stocks=20)
candidate_universe_20.to_csv(OUT_DIR / "candidate_universe_20.csv", index=False)
display(candidate_universe_20)

if not bin_df.empty:
    bin_df["month"] = bin_df["date_parsed"].dt.to_period("M").astype("string")
    rows_per_stock_month = bin_df.groupby(["stock", "month"]).size().rename("rows").reset_index()
    display(rows_per_stock_month.sort_values("rows", ascending=False).head(20))

# 8. Price and return checks

The synthetic alpha uses `P_t`; this section checks whether `mid` behaves like a stable price series and whether future returns can be computed within stock-date groups without leakage across days.

In [ ]:
price_horizons = [1, 5, 10, 30]
if not bin_df.empty:
    bin_returns = bin_df.copy().sort_values(["stock", "date_parsed", "timestamp"]).reset_index(drop=True)
    bin_returns["ret_mid_1"] = bin_returns.groupby(["stock", "date_parsed"], sort=False)["mid"].pct_change()
    if "midEnd" in bin_returns.columns:
        bin_returns["ret_midEnd"] = bin_returns["midEnd"] / bin_returns["mid"] - 1.0
    bin_returns = dau.add_future_returns(bin_returns, price_horizons, price_col="mid")

    for h in [1, 5, 10]:
        dau.plot_histogram(bin_returns[f"future_return_h{h}"], f"binSamples future return h={h}", f"future_return_h{h}", FIG_DIR / f"bin_return_hist_h{h}.png")
    dau.plot_sample_paths(bin_returns, FIG_DIR / "bin_mid_sample_paths.png", SELECTED_STOCKS, SAMPLE_N_STOCKS_FOR_PLOTS, SAMPLE_N_DAYS_FOR_PLOTS)

    large_price_jumps = dau.large_price_jumps(bin_returns)
    large_price_jumps.to_csv(OUT_DIR / "large_price_jumps.csv", index=False)
    display(large_price_jumps[["date", "time", "stock", "mid", "ret_mid_1", "threshold", "source_file"]].head(50) if not large_price_jumps.empty else large_price_jumps)

    return_cols = ["ret_mid_1", "ret_midEnd"] + [f"future_return_h{h}" for h in price_horizons if f"future_return_h{h}" in bin_returns]
    display(dau.numeric_summary(bin_returns[return_cols]))
else:
    bin_returns = pd.DataFrame()

# 9. Spread, depth, and liquidity checks

Liquidity variables determine whether impact estimates and trading constraints are economically plausible. This section studies spread, effective spread, depth, and their relationships.

In [ ]:
if not bin_df.empty:
    dau.plot_histogram(bin_df["spread"], "binSamples spread distribution", "spread", FIG_DIR / "bin_spread_hist.png")
    dau.plot_histogram(bin_df["effSpread"], "binSamples effective spread distribution", "effSpread", FIG_DIR / "bin_effspread_hist.png")
    dau.plot_histogram(bin_df["depth"], "binSamples depth distribution", "depth", FIG_DIR / "bin_depth_hist.png")
    dau.scatter_or_hexbin(bin_df, "spread", "depth", "binSamples spread vs depth", FIG_DIR / "bin_spread_depth_scatter.png")
    display(bin_df[[c for c in ["spread", "effSpread", "depth", "trdLiq", "ofLiq", "trade", "orderFlow"] if c in bin_df]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

if not fill_df.empty:
    dau.plot_histogram(fill_df["spread"], "fillSamples spread distribution", "spread", FIG_DIR / "fill_spread_hist.png")
    dau.plot_histogram(fill_df["effSpread"], "fillSamples effective spread distribution", "effSpread", FIG_DIR / "fill_effspread_hist.png")
    dau.plot_histogram(fill_df["depth"], "fillSamples depth distribution", "depth", FIG_DIR / "fill_depth_hist.png")
    dau.scatter_or_hexbin(fill_df, "spread", "depth", "fillSamples spread vs depth", FIG_DIR / "fill_spread_depth_scatter.png")
    display(fill_df[[c for c in ["spread", "effSpread", "depth", "trade", "askVolume", "bidVolume"] if c in fill_df]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

def plot_spread_depth_timeseries(df, path, title):
    if df.empty or not {"timestamp", "spread", "depth"}.issubset(df.columns):
        return
    stocks = SELECTED_STOCKS or sorted(df["stock"].dropna().unique())[:1]
    subset = df.loc[df["stock"].isin(stocks)].copy()
    dates = sorted(subset["date_parsed"].dropna().unique())[:1]
    subset = subset.loc[subset["date_parsed"].isin(dates)].copy()
    if subset.empty:
        return
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.plot(subset["timestamp"], subset["spread"], color="tab:blue", linewidth=1, label="spread")
    ax1.set_ylabel("spread", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(subset["timestamp"], subset["depth"], color="tab:orange", linewidth=1, alpha=0.8, label="depth")
    ax2.set_ylabel("depth", color="tab:orange")
    ax1.set_title(title)
    ax1.set_xlabel("timestamp")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

plot_spread_depth_timeseries(bin_df, FIG_DIR / "bin_spread_depth_timeseries.png", "binSamples spread and depth over one stock-day")
plot_spread_depth_timeseries(fill_df, FIG_DIR / "fill_spread_depth_timeseries.png", "fillSamples spread and depth over one stock-day")

# 10. Order flow and trade checks

Impact fitting depends on trade and order-flow variables. Here we inspect their distributions and simple predictive correlations with future returns.

In [ ]:
if not bin_returns.empty:
    dau.plot_histogram(bin_returns["orderFlow"], "binSamples orderFlow distribution", "orderFlow", FIG_DIR / "bin_orderflow_hist.png")
    dau.plot_histogram(bin_returns["trade"], "binSamples trade distribution", "trade", FIG_DIR / "bin_trade_hist.png")
    dau.scatter_or_hexbin(bin_returns, "orderFlow", "future_return_h5", "orderFlow vs future return h=5", FIG_DIR / "bin_orderflow_vs_future_return.png")
    dau.scatter_or_hexbin(bin_returns, "trade", "future_return_h5", "trade vs future return h=5", FIG_DIR / "bin_trade_vs_future_return.png")

    corr_cols = ["trade", "orderFlow", "lobImb", "effLobImb", "trdLiq", "ofLiq", "depth", "spread"] + [f"future_return_h{h}" for h in price_horizons]
    bin_correlation_table = dau.correlation_table(bin_returns, corr_cols)
    bin_correlation_table.to_csv(OUT_DIR / "bin_correlation_table.csv")
    display(bin_correlation_table)

    display(bin_returns[[c for c in ["trade", "orderFlow", "hidden", "auction", "nbEvents", "nbHidden", "nbTrades"] if c in bin_returns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# 11. LOB imbalance checks

LOB imbalance may be useful as a control or auxiliary predictor. We check whether it lies mostly in `[-1, 1]`, whether it correlates with future returns, and whether binned imbalance predicts average future return.

In [ ]:
lob_rows = []
for sample_name, df in [("bin", bin_returns), ("fill", fill_df)]:
    if df.empty or "lobImb" not in df.columns:
        continue
    future_col = "future_return_h5" if "future_return_h5" in df.columns else None
    in_range = df["lobImb"].between(-1, 1).mean()
    corr = float(df["lobImb"].corr(df[future_col])) if future_col else np.nan
    lob_rows.append({"sample_type": sample_name, "variable": "lobImb", "pct_in_minus1_1": in_range, "corr_future_return_h5": corr})
    dau.plot_histogram(df["lobImb"], f"{sample_name} lobImb distribution", "lobImb", FIG_DIR / f"{sample_name}_lobimb_hist.png")

if not bin_returns.empty and {"lobImb", "future_return_h5"}.issubset(bin_returns.columns):
    lob_binned = dau.binned_average(bin_returns, "lobImb", "future_return_h5", n_bins=10)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(lob_binned["x_mean"], lob_binned["y_mean"], marker="o")
    ax.set_title("Binned future return by lobImb")
    ax.set_xlabel("Average lobImb bin")
    ax.set_ylabel("Average future_return_h5")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "bin_lobimb_vs_future_return_binned.png", dpi=150)
    plt.close(fig)
    display(lob_binned)

lobimb_predictiveness = pd.DataFrame(lob_rows)
lobimb_predictiveness.to_csv(OUT_DIR / "lobimb_predictiveness.csv", index=False)
display(lobimb_predictiveness)

# 12. Merge binSamples and fillSamples

This section diagnoses how the aggregate bin data and fill-level data relate. Exact time matching can be sparse if fill timestamps are sub-second while bins are regular intervals, so the merge report is diagnostic rather than assumed to be perfect.

In [ ]:
merged_df, merge_report, common_column_comparison = dau.merge_diagnostics(bin_df, fill_df)
merge_report.to_csv(OUT_DIR / "bin_fill_merge_report.csv", index=False)
common_column_comparison.to_csv(OUT_DIR / "bin_fill_common_column_comparison.csv", index=False)

display(merge_report)
display(common_column_comparison)

for col, filename in [("mid", "merged_mid_comparison.png"), ("spread", "merged_spread_comparison.png"), ("depth", "merged_depth_comparison.png")]:
    if not merged_df.empty and {f"{col}_bin", f"{col}_fill"}.issubset(merged_df.columns):
        dau.scatter_or_hexbin(merged_df, f"{col}_bin", f"{col}_fill", f"Merged {col}: bin vs fill", FIG_DIR / filename, sample_size=100_000)

if not bin_df.empty and not fill_df.empty:
    key = ["stock", "date_parsed", "time"]
    for name, df in [("bin", bin_df), ("fill", fill_df)]:
        dup_counts = df.groupby(key).size().sort_values(ascending=False)
        print(name, "max duplicates per key:", int(dup_counts.max()) if len(dup_counts) else 0)
        display(dup_counts.head(10).reset_index(name="n_duplicates"))

# 13. Synthetic alpha feasibility checks

This directly supports section 2.4. We check whether `binSamples` provide enough valid observations for `alpha_t^h = x r_t^h + y DeltaW_t^h / P_t` across multiple horizons and target correlations.

In [ ]:
if not bin_df.empty:
    synthetic_alpha_feasibility = dau.synthetic_alpha_feasibility(
        bin_df,
        horizons=[1, 5, 10, 30],
        rhos=[0.05, 0.10, 0.20, 0.30, 0.50],
        random_seed=42,
    )
else:
    synthetic_alpha_feasibility = pd.DataFrame()

synthetic_alpha_feasibility.to_csv(OUT_DIR / "synthetic_alpha_feasibility.csv", index=False)
display(synthetic_alpha_feasibility)

if not synthetic_alpha_feasibility.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for h, group in synthetic_alpha_feasibility.groupby("horizon"):
        ax.plot(group["rho_target"], group["empirical_corr"], marker="o", label=f"h={h}")
    ax.plot([0, 0.55], [0, 0.55], linestyle="--", color="black", linewidth=1, label="target line")
    ax.set_title("Synthetic alpha empirical correlation check")
    ax.set_xlabel("Target rho")
    ax.set_ylabel("Empirical corr(alpha, future return)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "synthetic_alpha_corr_check.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    synthetic_alpha_feasibility["alpha_std"].hist(ax=ax, bins=30)
    ax.set_title("Synthetic alpha standard deviation across horizon/rho settings")
    ax.set_xlabel("alpha_std")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "synthetic_alpha_distribution.png", dpi=150)
    plt.close(fig)

# 14. Strategy and stress-test feasibility checks

This checks whether row-wise signal delay, wrong-model stress testing, and forced liquidation at 12:00 are feasible with the available intraday timestamps.

In [ ]:
stress_test_feasibility = dau.stress_test_feasibility(bin_df, liquidation_time="12:00")
stress_test_feasibility.to_csv(OUT_DIR / "stress_test_feasibility.csv", index=False)
display(stress_test_feasibility.head())
if not stress_test_feasibility.empty:
    display(stress_test_feasibility["feasible_for_forced_liquidation"].value_counts(dropna=False).rename("n_stock_days"))
    display(stress_test_feasibility["number_of_rows"].describe())

# 15. Rolling-window feasibility checks

The rolling-window study is owned by the teammate, but these checks confirm that my modules can consume month-tagged outputs and that consecutive train/test months have common stock coverage.

In [ ]:
rolling_window_plan = dau.rolling_window_plan(bin_df, candidate_universe_20)
rolling_window_plan.to_csv(OUT_DIR / "rolling_window_plan.csv", index=False)
display(rolling_window_plan)

# 16. Final audit report

The final report summarizes whether the data is usable for the project and highlights what should be clarified before final integration.

In [ ]:
def yes_no(value):
    return "Yes" if bool(value) else "No"

bin_available = not bin_df.empty
fill_available = not fill_df.empty
candidate_list = candidate_universe_20["stock"].head(20).tolist() if not candidate_universe_20.empty else []
common_merge_pct = float(merge_report.loc[merge_report["metric"] == "bin_rows_matched_pct", "value"].iloc[0]) if not merge_report.empty and "bin_rows_matched_pct" in merge_report["metric"].values else 0.0
forced_feasible_pct = float(stress_test_feasibility["feasible_for_forced_liquidation"].mean()) if not stress_test_feasibility.empty else 0.0
best_horizons = []
if not synthetic_alpha_feasibility.empty:
    rho_050 = synthetic_alpha_feasibility.loc[synthetic_alpha_feasibility["rho_target"] == 0.50].copy()
    rho_050["corr_error"] = (rho_050["empirical_corr"] - rho_050["rho_target"]).abs()
    best_horizons = rho_050.sort_values(["missing_rate_horizon_truncation", "corr_error"])["horizon"].head(3).tolist()

report_lines = [
    "Final Data Audit Summary",
    "========================",
    f"1. Can binSamples be used for synthetic alpha construction? {yes_no(bin_available and not synthetic_alpha_feasibility.empty)}.",
    "2. Recommended P_t column: mid. midEnd can be used as an end-of-bin comparison or return diagnostic.",
    f"3. Can future returns be computed without mixing days? {yes_no(bin_available and 'date_parsed' in bin_df.columns and 'timestamp' in bin_df.columns)}; computations use groupby stock/date.",
    f"4. Enough stocks for 20-stock one-month baseline? {yes_no(len(candidate_list) >= 20)}; candidate count = {len(candidate_list)}.",
    f"5. Enough data for rolling-window extension? {yes_no(not rolling_window_plan.empty and rolling_window_plan['number_of_common_stocks'].max() >= 20 if not rolling_window_plan.empty else False)}.",
    f"6. Can binSamples and fillSamples merge cleanly? Exact key match rate for bin rows is {common_merge_pct:.2%}; sub-second fill times may require asof/bin-aligned matching.",
    f"7. Obvious quality issues: inspect schema/data-quality CSVs, missing effSpread fields, duplicate keys, and large price jumps in outputs/data_audit/.",
    f"8. Candidate 20 stocks: {', '.join(candidate_list)}.",
    f"9. Reasonable synthetic alpha horizons from this audit: {best_horizons if best_horizons else 'inspect synthetic_alpha_feasibility.csv'}.",
    f"10. Is a one-row/minute signal delay feasible? {yes_no(bin_available)}; delay is implemented by stock/date group shift.",
    f"11. Is forced liquidation at 12:00 feasible? {forced_feasible_pct:.2%} of stock-days have rows before and after 12:00.",
    "12. Clarify with teammate/professor: exact bin interval interpretation, whether fill times should be aligned by exact time or nearest/asof bin, final impact parameter format, and whether mid or midEnd should define execution/PnL in the final simulator.",
]

dau.write_final_report(OUT_DIR / "final_data_audit_summary.txt", report_lines)
print("\n".join(report_lines))

# Synthetic Alpha Grid for Section 2.4

This section uses the already-loaded `bin_df` and writes the row-level baseline alpha plus diagnostics for the full `(h, rho)` sensitivity grid. Horizons are clock-time minutes, not row counts. The baseline is `h=5 minutes`, `rho=0.10`, with baseline decay half-life `H=5 minutes`; `rho` and `H` are sensitivity axes, not optimized market parameters.


In [ ]:
from src.alpha import run_synthetic_alpha_grid_clock_time

alpha_outputs, alpha_diag = run_synthetic_alpha_grid_clock_time(
    bin_df,
    horizons_minutes=[1.0, 5.0, 10.0],
    rhos=[0.05, 0.10, 0.20, 0.30, 0.50],
    baseline_horizon_minutes=5.0,
    baseline_rho=0.10,
    decay_half_lives_minutes=[1.0, 5.0, 30.0, 60.0],
    baseline_decay_half_life_minutes=5.0,
    output_dir=PROJECT_ROOT / "outputs" / "alphas",
    random_seed=42,
    save_all_alpha_outputs=False,
    make_plots=True,
)

display(alpha_diag)

baseline_alpha = alpha_outputs["h5m_rho010"]
display(baseline_alpha.head())


## Synthetic Alpha Validation Report

This formats the alpha diagnostics into a report-ready table, creates clean validation figures, and writes a short text summary.


In [ ]:
from src.alpha_reporting import create_alpha_validation_report

clean_alpha_table = create_alpha_validation_report(
    PROJECT_ROOT / "outputs" / "alphas" / "synthetic_alpha_diagnostics.csv",
    PROJECT_ROOT / "outputs" / "alphas",
)

display(clean_alpha_table)
